# 2D Ising model

*thermodynamics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/ising_model)

A lattice of $\pm 1$ spins that prefer to align with their neighbours. The neighbour sum of every site is one conv2d with a cross-shaped kernel, and a checkerboard Metropolis update flips half the lattice in parallel. Below the critical temperature domains grow until the magnet orders; above it the spins stay disordered.

**Vector form**

$$E = -J\sum_{\langle ij,kl\rangle} s_{ij}\,s_{kl},\qquad P(\text{flip}) = \min\!\big(1,\ e^{-\Delta E / k_B T}\big),\qquad \Delta E_{ij} = 2J\,s_{ij}\,h_{ij}$$

**Index (tensor) form**

$$h_{ij} = K_{kl}\,s_{i+k,\,j+l},\qquad E = -\tfrac{J}{2}\,s_{ij}\,h_{ij},\qquad m = \tfrac{1}{N}\textstyle\sum_{ij} s_{ij}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $s_{ij}$ | `s` | (B, 1, L, L) | spin $\pm 1$ at row $i$, column $j$ — for a batch of $B$ temperatures (the 1 is the conv2d channel) |
| $K_{kl}$ | `[[0, 1, 0], [1, 0, 1], [0, 1, 0]]` | (1, 1, 3, 3) | convolution kernel picking the four nearest neighbours |
| $h_{ij}$ | `conv2d(pad(s, mode='circular'), K)` | (B, 1, L, L) | repeated $k, l$ ⇒ summed: neighbour sum, periodic edges |
| $\Delta E_{ij}$ | `2 * J * s * h` | (B, 1, L, L) | cost of flipping each spin on its own |
| $s_{ij}h_{ij}$ | `einsum('bcij,bcij->b', s, h)` | (B,) | repeated $i, j$ ⇒ summed: twice the bond energy (every bond is seen from both ends) |
| $T_b$ | `T[:, None, None, None]` | (B, 1, 1, 1) | one temperature per lattice, broadcast over the sites |

Units: $J = k_B = 1$, so $T_c = 2/\ln(1 + \sqrt2) \approx 2.269$. Three lattices run side by side in the batch: the chosen $T$, $T_c$ and $T = 3.5$. All start from random spins (infinite temperature) and are suddenly cooled. Updating every site at once would be wrong — neighbours would flip based on stale values — but sites of one checkerboard colour only touch the other colour, so each half can be updated in parallel. A quench to low $T$ may get stuck for a long time in two stripe-shaped domains, a real feature of the dynamics on a periodic lattice. On a finite lattice $\lvert m\rvert$ is not exactly zero above $T_c$, and near $T_c$ it fluctuates strongly (critical slowing down). $E/N$ runs from $-2$ (perfect order) to 0.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/ising_model/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    L = params["L"]
    T = tensors.temperatures(params["T"], dev)  # (B,)
    s0 = tensors.random_spins(len(T), L, dev)  # (B, 1, L, L)
    masks = tensors.checkerboard(L, dev)

    r = solve.solve(s0, T, masks, params["n_sweeps"])

    exact = equation.onsager_magnetization(T)  # (B,)
    tail = r["m"][-50:].abs().mean(dim=0)  # (B,)  |m| averaged over the last sweeps
    for b in range(len(T)):
        print(f"T = {T[b]:.3f}   |m| = {tail[b]:.3f}  (Onsager {exact[b]:.3f})   E/N = {r['E'][-1, b]:.3f}")
    print(f"T_c = {equation.T_C:.4f}")
    return plot.animate(r["spins"], r["m"], r["E"], T, out_dir / "ising_model.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/ising_model/equation.py
r"""Physics → tensor translation.

The 2D Ising model: spins s_{ij} = ±1 on an L × L square lattice (periodic edges) with energy

    E = -J \sum_{\langle ij, kl \rangle} s_{ij} s_{kl}        (sum over nearest-neighbour pairs)

Write h_{ij} for the sum of the four neighbours of site (i, j). That is a convolution with a cross-shaped kernel,

    h_{ij} = \sum_{kl} K_{kl} \, s_{i+k, j+l},     K = [[0, 1, 0], [1, 0, 1], [0, 1, 0]]

so conv2d computes it for every site at once (circular padding makes the edges periodic). Then

    E = -\tfrac{J}{2} \, s_{ij} h_{ij}         (repeated i, j summed; ½ because every bond is seen twice)
    m = \tfrac{1}{N} \sum_{ij} s_{ij}          (magnetisation per spin, N = L²)

Metropolis: flipping s_{ij} costs \Delta E_{ij} = 2 J s_{ij} h_{ij}; accept the flip with probability
min(1, e^{-\Delta E / T}) (units with k_B = 1). Sites of one checkerboard colour have no neighbours of the same
colour, so all of them can be updated together without disturbing each other's h_{ij}.

Onsager's exact result: below T_c = 2J / \ln(1 + \sqrt2) ≈ 2.269 J the spontaneous magnetisation is
|m| = (1 - \sinh^{-4}(2J/T))^{1/8}, and m = 0 above T_c.
"""

import math

import torch
import torch.nn.functional as F

T_C = 2 / math.log(1 + math.sqrt(2))  # critical temperature for J = 1, k_B = 1
KERNEL = torch.tensor([[0.0, 1.0, 0.0], [1.0, 0.0, 1.0], [0.0, 1.0, 0.0]])  # K_{kl}: the four neighbours


def neighbor_sum(s: torch.Tensor) -> torch.Tensor:
    """h_{ij} = Σ_kl K_kl s_{i+k, j+l} with periodic edges.   s: (B, 1, L, L) → (B, 1, L, L)"""
    padded = F.pad(s, (1, 1, 1, 1), mode="circular")  # (B, 1, L+2, L+2)  wrap the edges around
    kernel = KERNEL.to(s.device, s.dtype)[None, None]  # (1, 1, 3, 3)  out channels, in channels, 3, 3
    return F.conv2d(padded, kernel)


def energy_per_spin(s: torch.Tensor, J: float = 1.0) -> torch.Tensor:
    r"""E / N = −(J/2) s_{ij} h_{ij} / N.   (B, 1, L, L) → (B,)"""
    h = neighbor_sum(s)
    n_sites = s.shape[-1] * s.shape[-2]
    return -0.5 * J * torch.einsum("bcij,bcij->b", s, h) / n_sites


def magnetization(s: torch.Tensor) -> torch.Tensor:
    """m = average spin.   (B, 1, L, L) → (B,)"""
    return s.mean(dim=(1, 2, 3))


def flip_cost(s: torch.Tensor, h: torch.Tensor, J: float = 1.0) -> torch.Tensor:
    """ΔE_{ij} = 2 J s_{ij} h_{ij}: energy change if spin (i, j) alone were flipped.   (B, 1, L, L)"""
    return 2 * J * s * h


def onsager_magnetization(T: torch.Tensor, J: float = 1.0) -> torch.Tensor:
    """Exact spontaneous magnetisation of the infinite lattice: (1 − sinh(2J/T)⁻⁴)^(1/8) below T_c, else 0."""
    inside = 1 - torch.sinh(2 * J / T) ** -4
    return torch.where(T < T_C, inside.clamp(min=0) ** 0.125, torch.zeros_like(T))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/ising_model/tensors.py
"""Build the concrete tensors: temperatures, a random spin lattice per temperature, checkerboard masks."""

import torch

from .equation import T_C


def temperatures(T: float, device: torch.device) -> torch.Tensor:
    """The chosen temperature plus two references: at T_c and well above it.  Shape (B,) = (3,)."""
    return torch.tensor([T, T_C, 3.5], device=device)


def random_spins(n_batch: int, L: int, device: torch.device, seed: int = 0) -> torch.Tensor:
    """s_{ij} = ±1 with equal chance ("infinitely hot" start).  Shape (B, 1, L, L): batch, channel, rows, cols."""
    torch.manual_seed(seed)  # fixed seed: the same picture every run
    return (2 * torch.randint(0, 2, (n_batch, 1, L, L)) - 1).float().to(device)


def checkerboard(L: int, device: torch.device) -> list[torch.Tensor]:
    """Two boolean masks of shape (L, L): sites with i + j even (black) and odd (white)."""
    i, j = torch.meshgrid(torch.arange(L), torch.arange(L), indexing="ij")
    black = ((i + j) % 2 == 0).to(device)
    return [black, ~black]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/ising_model/solve.py
"""Metropolis Monte Carlo with checkerboard updates: half the lattice flips in parallel, then the other half."""

import torch

from .equation import energy_per_spin, flip_cost, magnetization, neighbor_sum


def metropolis_sweep(s: torch.Tensor, T: torch.Tensor, masks: list[torch.Tensor], J: float = 1.0) -> torch.Tensor:
    """One sweep = one Metropolis attempt per site.   s: (B, 1, L, L), T: (B,)."""
    T = T[:, None, None, None]  # (B, 1, 1, 1)  one temperature per lattice in the batch
    for mask in masks:  # black sites, then white sites
        dE = flip_cost(s, neighbor_sum(s), J)  # (B, 1, L, L)
        accept = torch.rand_like(s) < torch.exp(-dE / T)  # always True when dE ≤ 0
        s = torch.where(mask & accept, -s, s)
    return s


def solve(s0: torch.Tensor, T: torch.Tensor, masks: list[torch.Tensor], n_sweeps: int, J: float = 1.0) -> dict:
    """Return snapshots (S+1, B, L, L) as int8, magnetisation and energy per spin (S+1, B)."""
    s = s0
    frames, m, E = [s0[:, 0].to(torch.int8)], [magnetization(s0)], [energy_per_spin(s0, J)]
    for _ in range(n_sweeps):
        s = metropolis_sweep(s, T, masks, J)
        frames.append(s[:, 0].to(torch.int8))
        m.append(magnetization(s))
        E.append(energy_per_spin(s, J))
    return {"spins": torch.stack(frames), "m": torch.stack(m), "E": torch.stack(E), "final": s}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/ising_model/plot.py
"""Animate the spin lattice at the chosen temperature beside |m| and E/N for every lattice in the batch."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import onsager_magnetization


def animate(spins: torch.Tensor, m: torch.Tensor, E: torch.Tensor, T: torch.Tensor, path: Path) -> Path:
    s = spins[:, 0].cpu().numpy()  # (S+1, L, L)  lattice at the chosen temperature
    m_abs = m.abs().cpu().numpy()  # (S+1, B)
    E_np = E.cpu().numpy()  # (S+1, B)
    temps = T.cpu().tolist()
    exact = onsager_magnetization(T).cpu().tolist()
    sweeps = np.arange(len(s))
    labels = [f"T = {temps[0]:.2f} (chosen)", f"T = {temps[1]:.2f} = T_c", f"T = {temps[2]:.2f}"]

    fig = plt.figure(figsize=(7, 4))
    ax = fig.add_subplot(1, 2, 1)
    ax_m = fig.add_subplot(2, 2, 2)
    ax_E = fig.add_subplot(2, 2, 4, sharex=ax_m)

    image = ax.imshow(s[0], cmap="coolwarm", vmin=-1, vmax=1, interpolation="nearest")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    title = ax.set_title("")

    lines_m, lines_E = [], []
    for b, label in enumerate(labels):
        lines_m.append(ax_m.plot([], [], color=ACCENT[b], lw=1.5, label=label)[0])
        lines_E.append(ax_E.plot([], [], color=ACCENT[b], lw=1.5)[0])
    ax_m.axhline(exact[0], color=ACCENT[0], ls=":", lw=1)  # Onsager's |m| for the chosen T
    ax_m.set_xlim(0, sweeps[-1])
    ax_m.set_ylim(0, 1.05)
    ax_m.set_ylabel("|m|")
    ax_m.legend(fontsize=7, loc="center right")
    ax_m.set_title("dotted: Onsager's exact |m|", fontsize=9)
    ax_E.set_ylim(-2.05, 0.05)
    ax_E.set_xlabel("Monte Carlo sweeps")
    ax_E.set_ylabel("E / N")
    plt.setp(ax_m.get_xticklabels(), visible=False)

    def update(step: int):
        image.set_data(s[step])
        for b in range(len(labels)):
            lines_m[b].set_data(sweeps[: step + 1], m_abs[: step + 1, b])
            lines_E[b].set_data(sweeps[: step + 1], E_np[: step + 1, b])
        title.set_text(f"T = {temps[0]:.2f}   sweep {step}   m = {m_abs[step, 0]:.2f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(s), 100), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'T': 1.8,
    'L': 96,
    'n_sweeps': 400
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.ising_model")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.ising_model.main")
problem_dir = Path("problems/thermodynamics/ising_model")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
